In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kangerz22/pa2sftcheckpoint/checkpoint.json
/kaggle/input/datasets/kangerz22/pa2sftcheckpoint/config.json
/kaggle/input/datasets/kangerz22/pa2sftcheckpoint/merges.txt
/kaggle/input/datasets/kangerz22/pa2sftcheckpoint/metrics.jsonl
/kaggle/input/datasets/kangerz22/pa2sftcheckpoint/tokenizer.json
/kaggle/input/datasets/kangerz22/pa2sftcheckpoint/vocab.json
/kaggle/input/datasets/kangerz22/pa2sftcheckpoint/tokenizer_config.json
/kaggle/input/datasets/kangerz22/pa2sftcheckpoint/loss.png
/kaggle/input/datasets/kangerz22/pa2sftcheckpoint/summary.json
/kaggle/input/datasets/kangerz22/pa2sftcheckpoint/chat_template.jinja
/kaggle/input/datasets/kangerz22/pa2sftcheckpoint/model.safetensors
/kaggle/input/datasets/kangerz22/pa2sftcheckpoint/special_tokens_map.json
/kaggle/input/datasets/kangerz22/pa2sftcheckpoint/configuration.json
/kaggle/input/datasets/kangerz22/pa2sftcheckpoint/added_tokens.json
/kaggle/input/datasets/kangerz22/pa2sftcheckpoint/generation_config.json
/kagg

In [2]:
%pip install -q "datasets>=2.18,<5" "transformers>=4.43,<5" sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 62.2 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import csv
import hashlib
import json
import zipfile

from pathlib import Path

from datasets import load_from_disk
from transformers import AutoTokenizer


inputroot = Path("/kaggle/input")
outputdirectory = Path("/kaggle/working/truncation_audit")
outputdirectory.mkdir(parents=True, exist_ok=True)


def finduniquefolder(foldername):
    matches = []

    for candidate in inputroot.rglob(foldername):
        if (
            candidate.is_dir()
            and (candidate / "dataset").is_dir()
            and (candidate / "manifest.json").is_file()
        ):
            matches.append(candidate)

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one original folder named {foldername}, "
            f"but found {len(matches)}: {[str(path) for path in matches]}. "
            "Attach the original DPO training notebook outputs."
        )

    return matches[0]


cachepaths = {
    "hh": finduniquefolder("reference_hh_full"),
    "ultrafeedback": finduniquefolder(
        "reference_ultrafeedback_full"
    ),
}

tokenizercandidates = []

for candidate in inputroot.rglob("tokenizer_config.json"):
    if "pa2sftcheckpoint" in str(candidate).lower():
        tokenizercandidates.append(candidate.parent)

if len(tokenizercandidates) != 1:
    raise RuntimeError(
        "Expected exactly one saved tokenizer inside pa2sftcheckpoint, "
        f"but found {len(tokenizercandidates)}: "
        f"{[str(path) for path in tokenizercandidates]}. "
        "Attach the SFT checkpoint dataset and check its folder structure."
    )

tokenizerpath = tokenizercandidates[0]

tokenizer = AutoTokenizer.from_pretrained(
    str(tokenizerpath),
    local_files_only=True,
    fix_mistral_regex=False,
)

if tokenizer.eos_token_id is None:
    raise RuntimeError(
        "The saved tokenizer has no EOS token ID. "
        "The audit cannot reconstruct the response-ending token."
    )

print("Tokenizer:", tokenizerpath)
print("EOS token ID:", tokenizer.eos_token_id)

for datasetname, cachepath in cachepaths.items():
    print(f"\nDataset: {datasetname}")
    print("Original reference cache:", cachepath)

    dataset = load_from_disk(str(cachepath / "dataset"))

    print("Splits:", list(dataset.keys()))

    for splitname in dataset.keys():
        print(
            f"{splitname}: {len(dataset[splitname])} pairs; "
            f"columns: {dataset[splitname].column_names}"
        )

    manifest = json.loads(
        (cachepath / "manifest.json").read_text(
            encoding="utf-8"
        )
    )

    (outputdirectory / f"{datasetname}_source_manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

Tokenizer: /kaggle/input/datasets/kangerz22/pa2sftcheckpoint
EOS token ID: 151643

Dataset: hh
Original reference cache: /kaggle/input/notebooks/kangerz22/pa2-dpotraining/pa2-small-model-alignment/outputs/reference_hh_full
Splits: ['train', 'validation']
train: 15000 pairs; columns: ['instruction_hash', 'instruction', 'prompt', 'chosen_response', 'rejected_response', 'chosen', 'rejected', 'chosen_response_token_count', 'rejected_response_token_count', 'reference_chosen_logps', 'reference_rejected_logps']
validation: 1000 pairs; columns: ['instruction_hash', 'instruction', 'prompt', 'chosen_response', 'rejected_response', 'chosen', 'rejected', 'chosen_response_token_count', 'rejected_response_token_count', 'reference_chosen_logps', 'reference_rejected_logps']

Dataset: ultrafeedback
Original reference cache: /kaggle/input/notebooks/kangerz22/pa2-dpotraining/pa2-small-model-alignment/outputs/reference_ultrafeedback_full
Splits: ['train', 'validation']
train: 15000 pairs; columns: ['instr

In [6]:
def auditresponse(record, side):
    responsefield = f"{side}_response"

    if responsefield not in record:
        raise KeyError(
            f"The saved row is missing {responsefield}. "
            f"Available fields: {list(record.keys())}."
        )

    if side not in record or not isinstance(record[side], dict):
        raise ValueError(
            f"The saved row has no tokenized {side} dictionary."
        )

    response = record[responsefield]
    batch = record[side]

    if not isinstance(response, str):
        raise TypeError(
            f"{responsefield} must contain text, "
            f"but received {type(response).__name__}."
        )

    for field in ["input_ids", "labels"]:
        if field not in batch:
            raise KeyError(
                f"The {side} dictionary is missing {field}. "
                f"Available fields: {list(batch.keys())}."
            )

    inputtokens = list(batch["input_ids"])
    labels = list(batch["labels"])

    if len(inputtokens) != len(labels):
        raise ValueError(
            f"The {side} input IDs and labels have different lengths."
        )

    attentionmask = batch.get("attention_mask")

    if attentionmask is not None:
        attentionmask = list(attentionmask)

        if len(attentionmask) != len(inputtokens):
            raise ValueError(
                f"The {side} attention mask and input IDs "
                "have different lengths."
            )

    retainedtokens = []
    responsepositions = []

    for position, label in enumerate(labels):
        if label == -100:
            continue

        if label != inputtokens[position]:
            raise ValueError(
                f"The {side} label at position {position} "
                "does not match its input token."
            )

        if (
            attentionmask is not None
            and attentionmask[position] != 1
        ):
            raise ValueError(
                f"The {side} response token at position {position} "
                "has a masked attention position."
            )

        retainedtokens.append(label)
        responsepositions.append(position)

    if responsepositions:
        firstresponseposition = responsepositions[0]
        lastresponseposition = responsepositions[-1]

        if responsepositions != list(
            range(firstresponseposition, lastresponseposition + 1)
        ):
            raise ValueError(
                f"The {side} response labels are not contiguous. "
                "The audit requires one continuous response span."
            )

        if attentionmask is None:
            activeinputtokens = lastresponseposition + 1
            retainedprompttokens = firstresponseposition
        else:
            activeinputtokens = sum(attentionmask)
            retainedprompttokens = sum(
                attentionmask[:firstresponseposition]
            )
    else:
        activeinputtokens = (
            sum(attentionmask)
            if attentionmask is not None
            else len(inputtokens)
        )
        retainedprompttokens = activeinputtokens

    responsetokens = tokenizer.encode(
        response,
        add_special_tokens=False,
    )

    expectedtokens = responsetokens + [tokenizer.eos_token_id]

    prefixmatches = (
        len(retainedtokens) <= len(expectedtokens)
        and retainedtokens == expectedtokens[:len(retainedtokens)]
    )

    endedwitheos = (
        bool(retainedtokens)
        and retainedtokens[-1] == tokenizer.eos_token_id
    )

    truncated = None
    contenttruncated = None
    onlyeosremoved = None
    removedtokens = None
    retentionfraction = None

    if prefixmatches:
        removedtokens = len(expectedtokens) - len(retainedtokens)
        truncated = removedtokens > 0
        contenttruncated = len(retainedtokens) < len(responsetokens)
        onlyeosremoved = (
            truncated
            and len(retainedtokens) == len(responsetokens)
        )
        retentionfraction = (
            len(retainedtokens) / len(expectedtokens)
        )

    return {
        "full_response_tokens_with_eos": len(expectedtokens),
        "full_response_content_tokens": len(responsetokens),
        "retained_response_tokens": len(retainedtokens),
        "predictable_response_tokens": sum(
            position > 0 for position in responsepositions
        ),
        "active_sequence_tokens": activeinputtokens,
        "retained_prompt_tokens": retainedprompttokens,
        "prefix_matches_reconstructed_response": prefixmatches,
        "truncated": truncated,
        "response_content_truncated": contenttruncated,
        "only_eos_removed": onlyeosremoved,
        "removed_response_tokens": removedtokens,
        "response_retention_fraction": retentionfraction,
        "ends_with_eos": endedwitheos,
        "empty_retained_response": not retainedtokens,
    }

In [7]:
allsummaries = []

for datasetname, cachepath in cachepaths.items():
    dataset = load_from_disk(str(cachepath / "dataset"))

    for splitname in dataset.keys():
        split = dataset[splitname]

        if len(split) == 0:
            raise ValueError(
                f"{datasetname}/{splitname} contains no preference pairs."
            )

        rows = []
        mismatchexamples = []

        print(
            f"\nAuditing {datasetname}/{splitname}: "
            f"{len(split)} pairs",
            flush=True,
        )

        for rownumber, record in enumerate(split):
            instruction = record.get(
                "instruction",
                record.get("prompt", ""),
            )

            row = {
                "dataset": datasetname,
                "split": splitname,
                "row_number": rownumber,
                "instruction_sha256": hashlib.sha256(
                    instruction.encode("utf-8")
                ).hexdigest(),
            }

            for side in ["chosen", "rejected"]:
                statistics = auditresponse(record, side)

                for name, value in statistics.items():
                    row[f"{side}_{name}"] = value

                if (
                    not statistics[
                        "prefix_matches_reconstructed_response"
                    ]
                    and len(mismatchexamples) < 10
                ):
                    mismatchexamples.append(
                        {
                            "row_number": rownumber,
                            "side": side,
                            "instruction": instruction,
                            "response": record[f"{side}_response"],
                            "statistics": statistics,
                            "retained_response_token_ids": [
                                label
                                for label in record[side]["labels"]
                                if label != -100
                            ],
                            "reconstructed_response_token_ids": (
                                tokenizer.encode(
                                    record[f"{side}_response"],
                                    add_special_tokens=False,
                                )
                                + [tokenizer.eos_token_id]
                            ),
                        }
                    )

            rows.append(row)

            if (rownumber + 1) % 2500 == 0:
                print(
                    f"Audited {rownumber + 1}/{len(split)} pairs",
                    flush=True,
                )

        verifiedpairs = []

        for row in rows:
            if (
                row["chosen_prefix_matches_reconstructed_response"]
                and row["rejected_prefix_matches_reconstructed_response"]
            ):
                verifiedpairs.append(row)

        pairgroups = {
            "neither_truncated": 0,
            "chosen_only_truncated": 0,
            "rejected_only_truncated": 0,
            "both_truncated": 0,
        }

        for row in verifiedpairs:
            chosentruncated = row["chosen_truncated"]
            rejectedtruncated = row["rejected_truncated"]

            if chosentruncated and rejectedtruncated:
                pairgroups["both_truncated"] += 1
            elif chosentruncated:
                pairgroups["chosen_only_truncated"] += 1
            elif rejectedtruncated:
                pairgroups["rejected_only_truncated"] += 1
            else:
                pairgroups["neither_truncated"] += 1

        summary = {
            "dataset": datasetname,
            "split": splitname,
            "source_cache": str(cachepath),
            "total_pairs": len(rows),
            "verified_pairs": len(verifiedpairs),
            "pairs_with_tokenization_mismatch": (
                len(rows) - len(verifiedpairs)
            ),
            "reconstruction_policy": (
                "Encode saved response without special tokens, "
                "then append one EOS token."
            ),
            "chosen": summarizeresponses(rows, "chosen"),
            "rejected": summarizeresponses(rows, "rejected"),
            "pair_truncation_groups_among_verified": pairgroups,
        }

        allsummaries.append(summary)

        detailpath = (
            outputdirectory
            / f"{datasetname}_{splitname}_details.csv"
        )

        with detailpath.open(
            "w",
            newline="",
            encoding="utf-8",
        ) as destination:
            writer = csv.DictWriter(
                destination,
                fieldnames=list(rows[0].keys()),
            )

            writer.writeheader()
            writer.writerows(rows)

        mismatchpath = (
            outputdirectory
            / f"{datasetname}_{splitname}_mismatches.json"
        )

        mismatchpath.write_text(
            json.dumps(
                mismatchexamples,
                indent=2,
                ensure_ascii=False,
                allow_nan=False,
            ),
            encoding="utf-8",
        )

        print(json.dumps(summary, indent=2))
        print("Saved detailed audit:", detailpath)

summarypath = outputdirectory / "truncation_summary.json"

summarypath.write_text(
    json.dumps(
        allsummaries,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    ),
    encoding="utf-8",
)

print("\nSaved combined summary:", summarypath)


Auditing hh/train: 15000 pairs
Audited 2500/15000 pairs
Audited 5000/15000 pairs
Audited 7500/15000 pairs
Audited 10000/15000 pairs
Audited 12500/15000 pairs
Audited 15000/15000 pairs
{
  "dataset": "hh",
  "split": "train",
  "source_cache": "/kaggle/input/notebooks/kangerz22/pa2-dpotraining/pa2-small-model-alignment/outputs/reference_hh_full",
  "total_pairs": 15000,
  "verified_pairs": 15000,
  "pairs_with_tokenization_mismatch": 0,
  "reconstruction_policy": "Encode saved response without special tokens, then append one EOS token.",
  "chosen": {
    "total_responses": 15000,
    "verified_response_prefixes": 15000,
    "tokenization_mismatches": 0,
    "responses_missing_eos": 9,
    "empty_retained_responses": 0,
    "truncated_responses": 9,
    "responses_with_content_removed": 9,
    "responses_with_only_eos_removed": 0,
    "mean_full_response_tokens_with_eos": 89.39573333333334,
    "mean_retained_response_tokens": 89.34186666666666,
    "truncated_percent_among_verified": 